# Pandas pour ingénieurs
**Synthèse des notions indispensables (Chapitre 3)**  
*Cycle Ingénieur / M1 — Focus sur les séries temporelles*

## Objectifs

À l’issue de ce chapitre, vous serez capable de :

- Manipuler les objets fondamentaux : **Series** et **DataFrame**
- Sélectionner des données avec `loc`, `iloc` et des masques booléens
- Comprendre l’**alignement d’index** et ses effets sur les opérations vectorisées
- Gérer les **valeurs manquantes** et nettoyer des données réelles
- Agréger efficacement avec `groupby` et des fonctions d’agrégation
- Joindre des tables (`concat`, `merge`) avec les bons réflexes
- Maîtriser l’essentiel des **séries temporelles** (objectif central du chapitre)


## Plan

1. Objets clés : **Series** & **DataFrame**
2. Indexation et sélection (`loc`, `iloc`, masques)
3. Alignement, opérations vectorisées, valeurs manquantes
4. Agrégation et `groupby`
5. Concaténation, `merge`, pivot
6. Chaînes de caractères et nettoyage rapide
7. **Séries temporelles (important)**
8. Synthèse & bonnes pratiques

## Pourquoi Pandas ?

NumPy est excellent pour le calcul numérique sur des tableaux homogènes.  
Pandas est conçu pour manipuler des **données tabulaires réelles** :

- Colonnes hétérogènes (nombres, texte, dates)
- Indexation par labels
- Gestion intégrée des valeurs manquantes
- Outils puissants pour les **séries temporelles**

 Dans un projet data/IA, une grande partie du travail consiste à préparer les données : c’est le rôle de Pandas.


# 1. Objets clés : Series & DataFrame

## Series

Une **Series** est un objet Pandas **unidimensionnel** qui associe :

- un **tableau de valeurs**
- un **index explicite** (étiquettes)

On peut la voir comme :
- un tableau NumPy **avec des labels**
- ou une **colonne de table** avec un index


In [1]:
import numpy as np
import pandas as pd

s = pd.Series([3.2, 4.1, 2.8], index=['capteur_A', 'capteur_B', 'capteur_C'])
print(s)
print('index:', s.index)
print('values:', s.values)

capteur_A    3.2
capteur_B    4.1
capteur_C    2.8
dtype: float64
index: Index(['capteur_A', 'capteur_B', 'capteur_C'], dtype='object')
values: [3.2 4.1 2.8]


## DataFrame

Un **DataFrame** est un objet Pandas **bidimensionnel** qui associe :

- un **index de lignes**
- des **colonnes nommées**
- des **données potentiellement hétérogènes**

On peut le voir comme :
- une **table de base de données**
- une **feuille Excel**
- un ensemble cohérent de **Series partageant le même index**


In [3]:
import pandas as pd

df = pd.DataFrame(
    {
        'temperature': [22, 23, 21],
        'vibration': [0.12, 0.15, 0.10]
    },
    index=['t1', 't2', 't3']
)

print(df)
print("\ndtypes :")
print(df.dtypes)


    temperature  vibration
t1           22       0.12
t2           23       0.15
t3           21       0.10

dtypes :
temperature      int64
vibration      float64
dtype: object


# 2. Indexation et sélection


## `loc` / `iloc`

Pandas propose **deux logiques d’indexation explicites** :

- `loc` : accès par **étiquette** (labels)
- `iloc` : accès par **position entière**


In [13]:
print(df.loc['t2', 'temperature'])  # accès par labels
print(df.iloc[1, 0])                # accès par positions


23
23


## Sélection par masque

Un **masque booléen** est une Series de valeurs `True / False`
utilisée pour filtrer les lignes d’un DataFrame.


In [4]:
mask = df['temperature'] > 22.5
print(df[mask])


    temperature  vibration
t2           23       0.15


# 3. Alignement, opérations, valeurs manquantes

## Alignement d’index

Lors d’une opération entre objets Pandas,
les calculs sont effectués **après alignement des index**.

In [4]:
a = pd.Series([1, 2, 3], index=['A', 'B', 'C'])
b = pd.Series([10, 20], index=['B', 'C'])
print(a)
print(b)
print(a + b)


A    1
B    2
C    3
dtype: int64
B    10
C    20
dtype: int64
A     NaN
B    12.0
C    23.0
dtype: float64


## Valeurs manquantes

Les données réelles contiennent souvent des **valeurs manquantes**,
représentées en Pandas par `NaN`.

Pandas fournit des méthodes dédiées pour :
- **repérer** les valeurs manquantes (`isna`)
- **traiter** ces valeurs (`fillna`, `dropna`)


In [5]:
print(df)
df_nan = df.copy()
df_nan.loc['t2', 'vibration'] = np.nan
print(df_nan)
print(df_nan.isna())
print(df_nan.fillna(df_nan.mean(numeric_only=True)))



    temperature  vibration
t1           22       0.12
t2           23       0.15
t3           21       0.10
    temperature  vibration
t1           22       0.12
t2           23        NaN
t3           21       0.10
    temperature  vibration
t1        False      False
t2        False       True
t3        False      False
    temperature  vibration
t1           22       0.12
t2           23       0.11
t3           21       0.10


# 4. Agrégation et `groupby`

## Agrégations simples
Pandas permet de réaliser rapidement des agrégations simples sur les colonnes numériques d’un DataFrame, comme :
- **moyenne**
- **écart-type**
- **min**, **max**, etc.

In [7]:
print(df.mean())          # Moyenne de chaque colonne numérique
print(df.agg(['min', 'max','mean']))  # Min et Max de chaque colonne


temperature    22.000000
vibration       0.123333
dtype: float64
      temperature  vibration
min          21.0   0.100000
max          23.0   0.150000
mean         22.0   0.123333


## GroupBy : split, apply, combine

Le mécanisme `groupby` repose sur trois étapes :
1. **split** : découper les données par catégorie
2. **apply** : appliquer une fonction à chaque groupe
3. **combine** : rassembler les résultats


In [6]:
rng = np.random.default_rng(0)

cat = pd.Series(rng.choice(['A', 'B'], size=6))
val = pd.Series(rng.normal(0, 1, size=6))
print(cat)
print(val)
ex = pd.DataFrame({'cat': cat, 'val': val})
print(ex)
print(ex.groupby('cat')['val'].mean())


0    B
1    B
2    B
3    A
4    A
5    A
dtype: str
0    0.104900
1   -0.535669
2    0.361595
3    1.304000
4    0.947081
5   -0.703735
dtype: float64
  cat       val
0   B  0.104900
1   B -0.535669
2   B  0.361595
3   A  1.304000
4   A  0.947081
5   A -0.703735
cat
A    0.515782
B   -0.023058
Name: val, dtype: float64


# 5. Concaténation, merge, pivot

## Concaténation

La concaténation permet **d’assembler plusieurs tables**
soit par **lignes**, soit par **colonnes**.

In [11]:
df1 = pd.DataFrame({'x': [1, 2], 'y': [3, 4]})
df2 = pd.DataFrame({'x': [5, 6], 'y': [7, 8]})

print(pd.concat([df1, df2], axis=0))

   x  y
0  1  3
1  2  4
0  5  7
1  6  8


## Merge / join

Le *merge* permet de **joindre des tables**
en faisant correspondre des lignes à partir d’une **clé commune**.


In [15]:
left = pd.DataFrame({'id': [1, 2, 3], 'temp': [20, 21, 19]})
right = pd.DataFrame({'id': [2, 3, 4], 'vib': [0.12, 0.09, 0.15]})
print(left)
print(right)
print(pd.merge(left, right, on='id', how='inner'))


   id  temp
0   1    20
1   2    21
2   3    19
   id   vib
0   2  0.12
1   3  0.09
2   4  0.15
   id  temp   vib
0   2    21  0.12
1   3    19  0.09


## Pivot table

Une *pivot table* permet de produire un **résumé multi-dimensionnel**
en réorganisant les données sous forme de tableau.


In [49]:
data = pd.DataFrame({
    'atelier': ['A', 'A', 'B', 'B'],
    'semaine': [1, 2, 1, 2],
    'defauts': [3, 4, 2, 5]
})
print(data)
print(
    pd.pivot_table(
        data,
        values='defauts',
        index='atelier',
        columns='semaine',
        aggfunc='mean'
    )
)


  atelier  semaine  defauts
0       A        1        3
1       A        2        4
2       B        1        2
3       B        2        5
semaine  1  2
atelier      
A        3  4
B        2  5


# 6. Chaînes et nettoyage rapide

## Opérations vectorisées
Les méthodes `str.*` permettent de réaliser des **opérations vectorisées**  
sur des chaînes de caractères pour nettoyer et normaliser des libellés.

In [52]:
s = pd.Series(['  Capteur-A ', 'capteur-b', 'CAPTEUR-c'])

print(
    s.str.strip()           # Supprimer les espaces en début et fin
    .str.lower()            # Mettre en minuscules
    .str.replace('-', '_')  # Remplacer les tirets par des underscores
)


0    capteur_a
1    capteur_b
2    capteur_c
dtype: str


# 7. Séries temporelles (important)

## Construction d’un index temporel

`pd.date_range` permet de construire un **index temporel régulier**,
adapté à la manipulation de séries temporelles.

In [18]:
idx = pd.date_range('2024-01-01', periods=6, freq='D')

ts = pd.Series([10, 11, 9, 12, 13, 8], index=idx)

print(ts)
print(ts.index)


2024-01-01    10
2024-01-02    11
2024-01-03     9
2024-01-04    12
2024-01-05    13
2024-01-06     8
Freq: D, dtype: int64
DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06'],
              dtype='datetime64[us]', freq='D')


## Sélection temporelle

Avec un index temporel, Pandas permet un **accès direct par période**
(année, mois, tranche de dates).


In [20]:
print(ts['2024-01-02':'2024-01-05'])

weekly = ts.resample('W').mean()
print(weekly)


2024-01-02    11
2024-01-03     9
2024-01-04    12
2024-01-05    13
Freq: D, dtype: int64
2024-01-07    10.5
Freq: W-SUN, dtype: float64


### Resampling : Changer la granularité
Le *resampling* permet de changer la **granularité temporelle** de la série :
- **jour → semaine**
- **semaine → mois**

In [19]:
weekly = ts.resample('W').mean()
print(weekly)
print(ts.rolling(window=3).mean())

2024-01-07    10.5
Freq: W-SUN, dtype: float64
2024-01-01          NaN
2024-01-02          NaN
2024-01-03    10.000000
2024-01-04    10.666667
2024-01-05    11.333333
2024-01-06    11.000000
Freq: D, dtype: float64
